# Introduction

This noteebook contains my attempt in creating a software able to forecast the winning team of a football game with machine learning models, using the odds of bookmakers and several performance statistics of the teams involved. 

This notebook has the porpouse to illustrate the complete workflow of this project, with detailed explenation of each chunk of code and of each decision made. 

The final software is created with specific Python scripts and it is thought to find valuable bets based on the predictions. 

In the description of this GitHub repository an explenation of the software usage is provided

In [1]:
import soccerdata as sd
import pandas as pd
import time 
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight
from skopt import BayesSearchCV, space
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, log_loss


[09/25/26 15:19:07] INFO     No custom team name replacements found. You can configure these in       ]8;id=508447;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=772037;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_config.py#92\92]8;;\
                             C:\Users\emanu\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=507433;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=427443;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_config.py#198\198]8;;\
                             C:\Users\emanu\soccerdata\config\league_dict.json.                                    

## 1. The Data

Since this is a Machine Learning problem the most important thing is gathering data that can solve the problem we are trying to solve. 
In this case it was decided to use data of each game from the 21-22 season to the 25-26 one of the 5 top European leagues, along with the the odds of every possible final result. 

This first chunck doesn't need to be run since it contains the code used to download all the training data, that I shared in the 'data' directory.

In [42]:
## let's start by getting all the games played in the season from 21-22 season
## in the european top 5 leagues and the key features of each game

leagues = ['ESP-La Liga', 'ITA-Serie A', 'ENG-Premier League', 'GER-Bundesliga', 'FRA-Ligue 1']
seasons = ['2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']

games_final = []

#collect all the data about games 
for league in leagues:
    games = sd.Understat(league, seasons)
    games_final.append(games.read_schedule())

    print('{} succesfully saved'.format(league))
    time.sleep(15)


games_df = pd.concat(games_final, axis=0)
games_df = games_df.reset_index()


games_df.to_csv('data/games.csv', index=False, sep=';')



#### And now let's import all the odds
'''
odds_final = []

for league in leagues: 
    for season in seasons:
        odds = sd.MatchHistory(leagues=league, seasons=season)

        odds_final.append(odds.read_games())
        print('{} {} succesfully saved'.format(league, season))
        time.sleep(5)


odds_df = pd.concat(odds_final, axis=0)
odds_df = odds_df.reset_index()

odds_df.to_csv('data/odds.csv', index=False, sep=';')

'''

### The previous code is the fastest way possible to collect data, but it may not work, 
## try manually downloading data from https://www.football-data.co.uk/ then execute the following code:

directory = Path('C:/Users/emanu/OneDrive/Desktop/progetti/value_betting_software/data/tmp') # write your own path

df_list = []

for file in directory.glob('*.csv'):
    df = pd.read_csv(file)
    df_list.append(df)

odds_df = pd.concat(df_list, axis=0, ignore_index=True)
odds_df = odds_df.drop_duplicates()
odds_df.to_csv('data/odds.csv', index=False)


[09/23/26 16:47:21] INFO     Saving cached data to C:\Users\emanu\soccerdata\data\Understat          ]8;id=846443;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=288070;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py#249\249]8;;\

[2026-09-23 16:47:21] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=136626;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=109497;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packa                 
                             ges\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll                          

ESP-La Liga succesfully saved


[09/23/26 16:47:37] INFO     Saving cached data to C:\Users\emanu\soccerdata\data\Understat          ]8;id=865429;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=286878;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py#249\249]8;;\

...

ENG-Premier League succesfully saved


[09/23/26 16:48:09] INFO     Saving cached data to C:\Users\emanu\soccerdata\data\Understat          ]8;id=532972;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=220899;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py#249\249]8;;\

GER-Bundesliga succesfully saved


[09/23/26 16:48:25] INFO     Saving cached data to C:\Users\emanu\soccerdata\data\Understat          ]8;id=666680;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=777301;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\soccerdata\_common.py#249\249]8;;\

FRA-Ligue 1 succesfully saved


We have several different dataframe, so it is time to merge them.

In [35]:
### In this section the data is merged in a single dataset and some preprocessing will be performed

games = pd.read_csv('data/games.csv', sep = ';')
odds = pd.read_csv('data/odds.csv')


## First let's select only the features that are needed

odds_filtered = odds.loc[:, ['Date', 'HomeTeam','AwayTeam', 'B365H', 'B365D', 'B365A', 'HS', 'AS', 'HST', 'AST', 'FTR', 'HC', 'AC']] 

games_filtered = games.loc[:, ['season', 'date', 'game', 'home_team','away_team','home_goals', 'away_goals', 'home_xg', 'away_xg']]


## Since the teams' names are not the same in the two df, let's address this problem 

games_teams = sorted(games_filtered['home_team'].unique())

odds_teams = sorted(odds_filtered['HomeTeam'].unique())

### The following dictionary contains all the different names, so we can replace them in the dataframe
different_names_dict = {
    'AC Milan': 'Milan',
    'Arminia Bielefeld': 'Bielefeld',
    'Athletic Club': 'Ath Bilbao',
    'Atletico Madrid': 'Ath Madrid',
    'Bayer Leverkusen': 'Leverkusen',
    'Borussia Dortmund': 'Dortmund',
    "Borussia M.Gladbach": "M'gladbach",
    'Celta Vigo': 'Celta',
    'Clermont Foot': 'Clermont',
    'Eintracht Frankfurt': 'Ein Frankfurt',
    'Espanyol': 'Espanol',
    'FC Cologne': 'FC Koln',
    'FC Heidenheim': 'Heidenheim',
    'Greuther Fuerth': 'Greuther Furth',
    'Hamburger SV': 'Hamburg',
    'Hertha Berlin': 'Hertha',
    'Mainz 05': 'Mainz',
    'Manchester City': 'Man City',
    'Manchester United': 'Man United',
    'Newcastle United': 'Newcastle',
    "Nottingham Forest": "Nott'm Forest",
    'Paris Saint Germain': 'Paris SG',
    'Parma Calcio 1913': 'Parma',
    'RasenBallsport Leipzig': 'RB Leipzig',
    'Rayo Vallecano': 'Vallecano',
    'Real Betis': 'Betis',
    'Real Oviedo': 'Oviedo',
    'Real Sociedad': 'Sociedad',
    'Real Valladolid': 'Valladolid',
    'Saint-Etienne': 'St Etienne',
    'St. Pauli': 'St Pauli',
    'VfB Stuttgart': 'Stuttgart',
    'Wolverhampton Wanderers': 'Wolves'
    }


games_filtered['home_team'] = games_filtered['home_team'].replace(different_names_dict)
games_filtered['away_team'] = games_filtered['away_team'].replace(different_names_dict)



## Let's aslso address the problem of the dates that are different in the 2 df
games_filtered['date'] = pd.to_datetime(games_filtered['date']).dt.normalize()

odds_filtered['Date'] = pd.to_datetime(odds_filtered['Date'], dayfirst=True).dt.normalize()


## And now finally merge the data in a single df
final_df = pd.merge(games_filtered, odds_filtered, left_on=['date', 'home_team','away_team'], right_on=['Date', 'HomeTeam','AwayTeam'])
final_df = final_df.drop_duplicates(subset=['date', 'home_team', 'away_team'])

### Now that the data has been cleared it's time for some feature engeneering to obtain all the final feature we want to include

[09/25/26 15:55:46] WARNING  C:\Users\emanu\AppData\Local\Temp\ipykernel_20624\4022246236.py:4:     ]8;id=123501;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py\warnings.py]8;;\:]8;id=210994;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py#110\110]8;;\
                             DtypeWarning: Columns (161) have mixed types. Specify dtype option on                 
                             import or set low_memory=False.                                                       
                               odds = pd.read_csv('data/odds.csv')                                                 
                                                                                                                   

In [44]:
final_df.head()

,season,date,game,home_team,away_team,home_goals,away_goals,home_xg,away_xg,Date,...,B365H,B365D,B365A,HS,AS,HST,AST,FTR,HC,AC
0,2122,2021-08-13,2021-08-13 Valencia-Getafe,Valencia,Getafe,1.0,0.0,1.578610,1.193260,2021-08-13,...,2.55,3.00,3.10,4.0,22.0,2.0,4.0,H,1.0,9.0
1,2122,2021-08-14,2021-08-14 Alaves-Real Madrid,Alaves,Real Madrid,1.0,4.0,1.410970,2.155510,2021-08-14,...,7.00,4.75,1.44,11.0,19.0,4.0,7.0,A,0.0,4.0
2,2122,2021-08-14,2021-08-14 Cadiz-Levante,Cadiz,Levante,1.0,1.0,0.993589,0.915954,2021-08-14,...,2.80,3.25,2.60,7.0,12.0,2.0,3.0,D,2.0,4.0
3,2122,2021-08-14,2021-08-14 Mallorca-Real Betis,Mallorca,Betis,1.0,1.0,0.569578,0.814085,2021-08-14,...,3.30,3.40,2.20,6.0,10.0,2.0,1.0,D,4.0,3.0
4,2122,2021-08-14,2021-08-14 Osasuna-Espanyol,Osasuna,Espanol,0.0,0.0,0.579404,0.583698,2021-08-14,...,2.25,3.20,3.40,14.0,10.0,1.0,3.0,D,4.0,6.0


And now that we have a final dataframe containing everything, let's perform some feature engineering to obtain all the features we need.

In [36]:
## First of all let's get the probability of every outcome (1/odd)

stakes = ['B365H', 'B365D', 'B365A']

for stake in stakes:
    final_df[stake] = 1/final_df[stake]

prov_sum = final_df.B365A + final_df.B365D + final_df.B365H

for stake in stakes:
    final_df[stake] = final_df[stake]/prov_sum


## Now we need to double every game so that we can work properly 
df_home = final_df[['season', 'date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_xg', 
                    'away_xg', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC']].copy()


df_home.columns = ['season', 'date', 'team', 'opponent', 'goals_for', 'goals_against', 'xG_for', 'xG_against', 
                   'shots_for', 'shots_against', 'shots_target_for', 'shots_target_against', 'corners_for', 'corners_against']

df_home['is_home'] = 1


df_away = final_df[['season', 'date', 'away_team', 'home_team', 'away_goals', 'home_goals', 
                    'away_xg', 'home_xg', 'AS', 'HS', 'AST', 'HST', 'AC', 'HC']].copy()

df_away.columns = ['season', 'date', 'team', 'opponent', 'goals_for', 'goals_against', 'xG_for', 'xG_against', 
                   'shots_for', 'shots_against', 'shots_target_for', 'shots_target_against', 'corners_for', 'corners_against']


df_away['is_home'] = 0



df_long = pd.concat([df_home, df_away], axis=0)


## And now lets's order the dataframe by team and date and compute all the feature that could be usefull:
## Rest days since lst match, total goals scored and conceded, total xG of the team and conceded, total points in the last 5 matches and
## points per game

df_long = df_long.sort_values(by=['team', 'date']).reset_index(drop=True)


mask = [df_long.goals_for > df_long.goals_against, 
        df_long.goals_for == df_long.goals_against,
        df_long.goals_for < df_long.goals_against]

df_long['points gained'] = np.select(mask, [3, 1, 0])

groups = df_long.groupby(['season', 'team'])

df_long['rest_days'] = groups['date'].diff().dt.days

df_long['total_goals'] = groups['goals_for'].transform(lambda x: x.cumsum().shift(1))

df_long['total_xg'] = groups['xG_for'].transform(lambda x: x.cumsum().shift(1))

df_long['total_goals_against'] = groups['goals_against'].transform(lambda x: x.cumsum().shift(1))

df_long['total_xg_against'] = groups['xG_against'].transform(lambda x: x.cumsum().shift(1))

df_long['last_5'] = groups['points gained'].transform(lambda x: x.shift(1).rolling(5).sum())

df_long['match_played'] = groups.cumcount()

df_long['PPG'] = groups['points gained'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_shots_for'] = groups['shots_for'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_shots_target_for'] = groups['shots_target_for'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_shots_against'] = groups['shots_against'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_shots_target_against'] = groups['shots_target_against'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_corners_for'] = groups['corners_for'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']

df_long['avg_corners_against'] = groups['corners_against'].transform(lambda x: x.cumsum().shift(1))/df_long['match_played']


############ Now it's time to merge this new features with the new ones

df = pd.merge(final_df, df_long, left_on=['date', 'home_team'], right_on=['date', 'team']).rename(columns={'goals_for':'home_goals_for',
                                                                                                           	'goals_against':'home_goals_against',
                                                                                                            'xG_for':'home_xG_for',	
                                                                                                            'xG_against':'home_xG_against'	,
                                                                                                            'is_home':'home_is_home',	
                                                                                                            'points gained':'home_points_gained',	
                                                                                                            'rest_days':'home_rest_days',	
                                                                                                            'total_goals':'home_total_goals',	
                                                                                                            'total_xg':'home_total_xg',	
                                                                                                            'total_goals_against':'home_total_goals_against',	
                                                                                                            'total_xg_against':'home_total_xg_against',	
                                                                                                            'last_5':'home_last_5',	
                                                                                                            'match_played':'home_match_played',	
                                                                                                            'PPG':'home_PPG',
                                                                                                            'avg_shots_for' : 'avg_home_shots',
                                                                                                            'avg_shots_target_for' : 'avg_home_shots_target',
                                                                                                            'avg_shots_against' : 'avg_home_shots_against',
                                                                                                            'avg_shots_target_against' : 'avg_home_target_shots_against',
                                                                                                            'avg_corners_for' : 'avg_home_corners_for',
                                                                                                            'avg_corners_against' : 'avg_home_corners_against'})
 
 
df = pd.merge(df, df_long, left_on=['date', 'away_team'], right_on=['date', 'team']).rename(columns={'goals_for':'away_goals_for',
                                                                                                           	'goals_against':'away_goals_against',
                                                                                                            'xG_for':'away_xG_for',	
                                                                                                            'xG_against':'away_xG_against'	,
                                                                                                            'is_home':'away_is_home',	
                                                                                                            'points gained':'away_points_gained',	
                                                                                                            'rest_days':'away_rest_days',	
                                                                                                            'total_goals':'away_total_goals',	
                                                                                                            'total_xg':'away_total_xg',	
                                                                                                            'total_goals_against':'away_total_goals_against',	
                                                                                                            'total_xg_against':'away_total_xg_against',	
                                                                                                            'last_5':'away_last_5',	
                                                                                                            'match_played':'away_match_played',	
                                                                                                            'PPG':'away_PPG',
                                                                                                            'avg_shots_for' : 'avg_away_shots',
                                                                                                            'avg_shots_target_for' : 'avg_away_shots_target',
                                                                                                            'avg_shots_against' : 'avg_away_shots_against',
                                                                                                            'avg_shots_target_against' : 'avg_away_target_shots_against',
                                                                                                            'avg_corners_for' : 'avg_away_corners_for',
                                                                                                            'avg_corners_against' : 'avg_away_corners_against'})


df = df.dropna()

df = df.reset_index(drop=True)


df.to_csv('data/final_data.csv', sep=';', index=False)

print('Data frame with shape {} correctly saved as CSV'.format(df.shape))


Data frame with shape (7636, 80) correctly saved as CSV


Now that all the matches are set, we can perform another round of feature engineering to compute differences between teams in goals, xG, results..

In [37]:
df['goals_for_difference'] = df['home_total_goals'] - df['away_total_goals']

df['goals_against_difference'] = df['home_total_goals_against'] - df['away_total_goals_against']

df['xG_for_difference'] = df['home_total_xg'] - df['away_total_xg']

df['xG_against_difference'] = df['home_total_xg_against'] - df['away_total_xg_against']

df['last_5_difference'] = df['home_last_5'] - df['away_last_5']

df['PPG_difference'] = df['home_PPG'] - df['away_PPG']

df.sort_values('date')

    
def elo_computing(df, k_factor=20, mean_reversion=0.25):
    # a function to compute elo ratings
    elo = {} 
    home_elos, away_elos = [], []
    current_season = None
    
    for index, row in df.iterrows():

        if current_season is not None and row['season'] != current_season:
            for team in elo:
                elo[team] = elo[team] * (1 - mean_reversion) + 1500.0 * mean_reversion
        
        current_season = row['season']
        home = row['home_team']
        away = row['away_team']

        #at the beginning it is set at 1500
        if home not in elo: elo[home] = 1500.0
        if away not in elo: elo[away] = 1500.0
        
        home_elos.append(elo[home])
        away_elos.append(elo [away])
        
        e_home = 1 / (1 + 10 ** ((elo[away] - elo[home]) / 400))
        e_away = 1 - e_home
        
        if row['home_goals'] > row['away_goals']:
            s_home, s_away = 1.0, 0.0
        elif row['home_goals'] == row['away_goals']:
            s_home, s_away = 0.5, 0.5
        else:
            s_home, s_away = 0.0, 1.0
            
        elo[home] = elo[home] + k_factor * (s_home - e_home)
        elo[away] = elo[away] + k_factor * (s_away - e_away)
        
    df['home_elo'] = home_elos
    df['away_elo'] = away_elos
    df['elo_difference'] = df['home_elo'] - df['away_elo']
    return df



df = elo_computing(df, 20, 0.25)

In [40]:
df.head()

,season_x,date,game,home_team,away_team,home_goals,away_goals,home_xg,away_xg,Date,...,avg_away_corners_against,goals_for_difference,goals_against_difference,xG_for_difference,xG_against_difference,last_5_difference,PPG_difference,home_elo,away_elo,elo_difference
0,2122,2021-09-21,2021-09-21 Athletic Club-Rayo Vallecano,Ath Bilbao,Vallecano,1.0,2.0,0.956027,1.041210,2021-09-21,...,4.0,-4.0,-4.0,-1.716242,-4.416140,2.0,0.4,1500.0,1500.0,0.0
1,2122,2021-09-21,2021-09-21 Getafe-Atletico Madrid,Getafe,Ath Madrid,1.0,2.0,0.330581,1.479060,2021-09-21,...,2.8,-6.0,4.0,-3.053585,2.813576,-11.0,-2.2,1500.0,1500.0,0.0
2,2122,2021-09-21,2021-09-21 Levante-Celta Vigo,Levante,Celta,0.0,2.0,1.407020,0.739683,2021-09-21,...,3.6,2.0,-3.0,-0.287713,-2.655334,3.0,0.6,1500.0,1500.0,0.0
3,2122,2021-09-22,2021-09-22 Real Madrid-Mallorca,Real Madrid,Mallorca,6.0,1.0,2.374600,1.191650,2021-09-22,...,2.8,12.0,4.0,6.203438,0.586769,5.0,1.0,1500.0,1500.0,0.0
4,2122,2021-09-23,2021-09-23 Granada-Real Sociedad,Granada,Sociedad,2.0,3.0,1.250190,1.581530,2021-09-23,...,3.8,-3.0,4.0,-4.592122,3.112262,-7.0,-1.4,1500.0,1500.0,0.0


## 2. The Predictive Model

Having all the data is crucial, now let's get to the funny stuff: it's time to find a nice model (a xgBoost model will  be trained).

After all the preprocessing and the feature engineering it's time to perform a little validation to find the best model and its best hyperparameters via a Bayes search. The test set will be composed by the totality of the 25/26 season, while the remaining part of the dataset will be the train set.

In [41]:
# extract the feature matrix
X = df[['season_x', 
       'B365H', 'B365D', 'B365A', 
       'home_is_home', 'home_rest_days', 'home_total_goals',
       'home_total_xg', 'home_total_goals_against', 'home_total_xg_against',
       'home_last_5', 'home_match_played', 'home_PPG', 'avg_home_shots',
       'avg_home_shots_target', 'avg_home_shots_against',
       'avg_home_target_shots_against', 'avg_home_corners_for',
       'avg_home_corners_against', 
       'away_is_home', 'away_rest_days', 'away_total_goals', 'away_total_xg',
       'away_total_goals_against', 'away_total_xg_against', 'away_last_5',
       'away_match_played', 'away_PPG', 'avg_away_shots',
       'avg_away_shots_target', 'avg_away_shots_against',
       'avg_away_target_shots_against', 'avg_away_corners_for',
       'avg_away_corners_against', 'last_5_difference', 'PPG_difference', 
       'home_elo', 'away_elo', 'elo_difference']]



# get the target feature
mask = [df.home_goals > df.away_goals,
        df.home_goals == df.away_goals,
        df.home_goals < df.away_goals]

df['final_result'] = np.select(mask, [0, 1, 2])

y = df[['final_result', 'season_x']]



#split train and test data
X_train = X[X['season_x'] != 2526]
X_train.drop('season_x', axis='columns', inplace=True) 

X_test  = X[X['season_x'] == 2526]
X_test.drop('season_x', axis='columns', inplace=True) 

y_train = y[y['season_x'] != 2526]
y_train.drop('season_x', axis='columns', inplace=True)

y_test = y[y['season_x'] == 2526]
y_test.drop('season_x', axis='columns', inplace=True)

[09/25/26 15:56:48] WARNING  C:\Users\emanu\AppData\Local\Temp\ipykernel_20624\459409549.py:33:     ]8;id=626979;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py\warnings.py]8;;\:]8;id=857829;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py#110\110]8;;\
                             SettingWithCopyWarning:                                                               
                             A value is trying to be set on a copy of a slice from a DataFrame                     
                                                                                                                   
                             See the caveats in the documentation:                                                 
                             https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#                
                             returning-a-view-versus-a-copy                                                        
                               X_train.drop('season_x', axis='columns', inplace=True)                              
                                                                                                                   

                    WARNING  C:\Users\emanu\AppData\Local\Temp\ipykernel_20624\459409549.py:36:     ]8;id=547019;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py\warnings.py]8;;\:]8;id=827061;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py#110\110]8;;\
                             SettingWithCopyWarning:                                                               
                             A value is trying to be set on a copy of a slice from a DataFrame                     
                                                                                                                   
                             See the caveats in the documentation:                                                 
                             https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#                
                             returning-a-view-versus-a-copy                                                        
                               X_test.drop('season_x', axis='columns', inplace=True)                               
                                                                                                                   

                    WARNING  C:\Users\emanu\AppData\Local\Temp\ipykernel_20624\459409549.py:39:     ]8;id=355293;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py\warnings.py]8;;\:]8;id=77847;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py#110\110]8;;\
                             SettingWithCopyWarning:                                                               
                             A value is trying to be set on a copy of a slice from a DataFrame                     
                                                                                                                   
                             See the caveats in the documentation:                                                 
                             https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#                
                             returning-a-view-versus-a-copy                                                        
                               y_train.drop('season_x', axis='columns', inplace=True)                              
                                                                                                                   

                    WARNING  C:\Users\emanu\AppData\Local\Temp\ipykernel_20624\459409549.py:42:     ]8;id=565239;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py\warnings.py]8;;\:]8;id=493129;file://c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\warnings.py#110\110]8;;\
                             SettingWithCopyWarning:                                                               
                             A value is trying to be set on a copy of a slice from a DataFrame                     
                                                                                                                   
                             See the caveats in the documentation:                                                 
                             https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#                
                             returning-a-view-versus-a-copy                                                        
                               y_test.drop('season_x', axis='columns', inplace=True)                               
                                                                                                                   

In [42]:
grid = {'max_depth': space.Integer(3, 7),
        'colsample_bytree': space.Real(0.6, 1),
        'learning_rate' : space.Real(1e-5, 1, prior = 'log-uniform'),
        'min_child_weight' : space.Integer(20, 100),
        'gamma' : space.Real(0.05, 5)}

model = xgb.XGBClassifier(objective='multi:softprob', eval_metric='mlogloss', random_state=62, device='cuda')
cv = TimeSeriesSplit(n_splits=5)

search = BayesSearchCV(model, grid, cv=cv, scoring='neg_log_loss', random_state=62, n_iter=40, verbose=3, n_jobs=1)

search.fit(X_train, y_train)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END colsample_bytree=0.7982897741985466, gamma=4.007952290736397, learning_rate=0.0003013506482415875, max_depth=6, min_child_weight=43;, score=-1.081 total time=   3.9s
[CV 2/5] END colsample_bytree=0.7982897741985466, gamma=4.007952290736397, learning_rate=0.0003013506482415875, max_depth=6, min_child_weight=43;, score=-1.076 total time=   5.1s
[CV 3/5] END colsample_bytree=0.7982897741985466, gamma=4.007952290736397, learning_rate=0.0003013506482415875, max_depth=6, min_child_weight=43;, score=-1.054 total time=   5.8s
[CV 4/5] END colsample_bytree=0.7982897741985466, gamma=4.007952290736397, learning_rate=0.0003013506482415875, max_depth=6, min_child_weight=43;, score=-1.072 total time=   5.7s
[CV 5/5] END colsample_bytree=0.7982897741985466, gamma=4.007952290736397, learning_rate=0.0003013506482415875, max_depth=6, min_child_weight=43;, score=-1.063 total time=   6.4s
Fitting 5 folds for each of 1 candidates, tota

,estimator,"XGBClassifier...ree=None, ...)"
,search_spaces,"{'colsample_bytree': Real(low=0.6,...m='normalize'), 'gamma': Real(low=0.05...m='normalize'), 'learning_rate': Real(low=1e-0...m='normalize'), 'max_depth': Integer(low=3...m='normalize'), ...}"
,optimizer_kwargs,None
,n_iter,40
,scoring,'neg_log_loss'
,fit_params,None
,n_jobs,1
,n_points,1
,iid,'deprecated'
,refit,True
,cv,TimeSeriesSpl...est_size=None)


In [43]:
best_hyperparameters = search.best_params_
print(best_hyperparameters)
print(search.best_score_)

OrderedDict([('colsample_bytree', 1.0), ('gamma', 4.144362139523036), ('learning_rate', 0.3014243840817577), ('max_depth', 3), ('min_child_weight', 39)])
-0.9801906448340224


### Test the model

Now that a possible best model had been found thanks to the validation, we can train a full model and test its accuracy and possible gains

In [44]:
model = xgb.XGBClassifier(objective='multi:softprob', random_state=62, device='cuda',
                          colsample_bytree=1, learning_rate=0.3014243840817577, max_depth=3, min_child_weight=39, 
                          gamma=4.144362139523036)

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)
preds = model.predict(X_test)

f1 = f1_score(y_test, preds, average='macro')
l_loss = log_loss(y_test, probs)



print('Macro f1 score: {}'.format(f1))
print('Logaritmic loss: {}'.format(l_loss))


Macro f1 score: 0.40264145927629685
Logaritmic loss: 0.9811567674830783


In [49]:
### Now that we have the model, let's see how much 100 euros would get us on the test set
initial_money = 100.0
current_money = initial_money
bets = 0
winning_bets = 0
th1 = 1.05
th2 = 0.5


for game in range(len(y_test)):

    game_index = y_test.index[game]

    home_odd = 1/df.loc[game_index, 'B365H']
    draw_odd =  1/df.loc[game_index, 'B365D']
    away_odd =  1/df.loc[game_index, 'B365A']

    home_win_prob = probs[game][0]
    draw_prob = probs[game][1]
    away_win_prob = probs[game][2]

    home_expected_value = home_odd * home_win_prob
    draw_expected_value = draw_odd * draw_prob
    away_expected_value = away_odd * away_win_prob

    bet = 5.0

    if home_expected_value >= th1 and home_win_prob >= th2:
        current_money -= bet
        bets += 1
        if y_test.iloc[game, 0] == 0:
            current_money += home_odd * bet
            winning_bets += 1

    elif away_expected_value >= th1 and away_win_prob >= th2:
        current_money -= bet
        bets += 1
        if y_test.iloc[game, 0] == 2:
            current_money += away_odd * bet
            winning_bets += 1

print('Final results:')
print('Total money: {:.2f}'.format(current_money))
print('Total bets won: {}'.format(winning_bets))
print('Total bets: {}'.format(bets))
# Corretto il calcolo dei soldi investiti
print('Money betted: {:.2f}'.format(bets * bet)) 
print('Revenue: {:.2f}'.format(current_money - initial_money))

if bets > 0:
    roi = ((current_money - initial_money) / (bets * bet)) * 100
    print(f'ROI: {roi:.2f}%')

Final results:
Total money: 186.78
Total bets won: 97
Total bets: 153
Money betted: 765.00
Revenue: 86.78
ROI: 11.34%
